In [1]:
import os
import pandas as pd
import regex as re
import utils as ut
import importlib as il
import numpy as np

#### constants and utils

In [2]:
metrics_or_logits = "logits"

r_tabnet_own = re.compile(f"^tabnet_own_.+\{metrics_or_logits}.csv$")
r_own = re.compile(f"^own_original.+\{metrics_or_logits}.csv$")
r_tabnet_paper = re.compile(f"^tabnet_paper_.+\{metrics_or_logits}.csv$")

In [3]:
file_names = {}
file_names["tabnet_own"] = list(filter(r_tabnet_own.match, os.listdir(ut.PATH_LOGS)))
file_names["own_original"] = list(filter(r_own.match, os.listdir(ut.PATH_LOGS)))
file_names["tabnet_paper"] = list(filter(r_tabnet_paper.match, os.listdir(ut.PATH_LOGS)))

file_names

{'tabnet_own': ['tabnet_own_comp_rs_unsup_logits.csv',
  'tabnet_own_comp_ss_logits.csv',
  'tabnet_own_comp_ss_sup_logits.csv',
  'tabnet_own_simp_ss_sup_logits.csv',
  'tabnet_own_simp_ss_unsup_logits.csv'],
 'own_original': ['own_original_comp_bal_fac_ss_logits.csv',
  'own_original_comp_fac_ss_logits.csv',
  'own_original_simp_fac_ss_logits.csv'],
 'tabnet_paper': ['tabnet_paper_rs_unsup_logits.csv',
  'tabnet_paper_ss_unsup_logits.csv']}

## COMPARE EXPERIMENTS

In [89]:
il.reload(ut)
metrics_or_logits = "logits"
res = ut.compare_experiments(["own_original_simp_fac_ss_logits","tabnet_own_simp_ss_unsup_logits","tabnet_own_simp_ss_sup_logits"],
                    aliases=["own_fac","tab_own_unsup","tab_own_sup"],title="resume",save=False)

res

,match,Date,season,Div,HomeTeam,AwayTeam,FTHG,FTAG,label,prediction_own_fac,...,prediction_tab_own_sup,draw_tab_own_sup,home_tab_own_sup,away_tab_own_sup,accurate_flag_own_fac,rps_own_fac,accurate_flag_tab_own_unsup,rps_tab_own_unsup,accurate_flag_tab_own_sup,rps_tab_own_sup
0,15751,2020-01-01,T19-20,E0,Brighton,Chelsea,1,1,0,2,...,2,0.2761,0.2447,0.4791,0,0.430418,0,0.378775,0,0.376832
1,15758,2020-01-01,T19-20,E0,West Ham,Bournemouth,4,0,1,1,...,1,0.2953,0.4460,0.2588,1,0.056424,1,0.074735,1,0.077064
2,15757,2020-01-01,T19-20,E0,Norwich,Crystal Palace,1,1,0,2,...,2,0.2643,0.2996,0.4361,0,0.389797,0,0.359225,0,0.365719
3,15756,2020-01-01,T19-20,E0,Man City,Everton,2,1,1,1,...,1,0.1621,0.7896,0.0483,1,0.006826,1,0.012099,1,0.014305
4,15759,2020-01-01,T19-20,E0,Arsenal,Man United,2,0,1,1,...,1,0.2677,0.3697,0.3626,1,0.086058,1,0.099856,1,0.101571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2626,26483,2021-05-23,T20-21,F1,Lens,Monaco,0,0,0,2,...,2,0.2790,0.2746,0.4464,0,0.387210,0,0.376541,0,0.359557
2627,26482,2021-05-23,T20-21,F1,Brest,Paris SG,0,2,2,2,...,2,0.1616,0.1006,0.7378,1,0.054306,1,0.065356,1,0.047432
2628,26481,2021-05-23,T20-21,F1,Angers,Lille,1,2,2,2,...,2,0.3039,0.2867,0.4094,1,0.116655,1,0.155165,1,0.220582
2629,26485,2021-05-23,T20-21,F1,Metz,Marseille,1,1,0,2,...,2,0.3095,0.3401,0.3504,0,0.396212,0,0.314969,0,0.299785


In [88]:
res.columns

Index(['match', 'Date', 'season', 'Div', 'HomeTeam', 'AwayTeam', 'FTHG',
       'FTAG', 'label', 'prediction_own_fac', 'draw_own_fac', 'home_own_fac',
       'away_own_fac', 'prediction_tab_own_unsup', 'draw_tab_own_unsup',
       'home_tab_own_unsup', 'away_tab_own_unsup', 'prediction_tab_own_sup',
       'draw_tab_own_sup', 'home_tab_own_sup', 'away_tab_own_sup',
       'accurate_flag_own_fac', 'rps_own_fac', 'accurate_flag_tab_own_unsup',
       'rps_tab_own_unsup', 'accurate_flag_tab_own_sup', 'rps_tab_own_sup'],
      dtype='object')

## RESUME METRICS

- Accuracy absoluto
- Accuracy relativo (haciendo inner join)
- RPS
- bias con empates (min 0.5 para home/away win)

### General accuracy

In [4]:
metrics_or_logits = "logits"

metricsDFs = ut.read_dataframes(["\w+metrics"])

In [339]:
metrics = { "validation": {},  "abs_test": {}, }
for key in metricsDFs.keys():
    # name = re.match(r'(\w+)_metrics',key).group(1)
    name = ut.filter_group(r'(\w+)_metrics',key)
    test_max = metricsDFs[key].test.round(4).max()
    val_max = metricsDFs[key].validation.round(4).max()
    metrics["abs_test"][name] = test_max
    metrics["validation"][name] = val_max

resume = pd.DataFrame(metrics).sort_values('abs_test',ascending=False)
resume

,validation,abs_test
tabnet_own_comp_ss,0.5327,0.5021
own_original_comp_fac_ss,0.5347,0.4998
own_original_simp_fac_ss,0.5312,0.4998
tabnet_own_simp_ss_unsup,0.5266,0.4964
tabnet_own_comp_ss_sup,0.5314,0.4941
tabnet_own_simp_ss_sup,0.5298,0.4941
tabnet_own_comp_rs_unsup,0.5220,0.4922
tabnet_paper_ss_unsup,0.5180,0.4857
tabnet_paper_rs_unsup,0.5063,0.4807
own_original_comp_bal_fac_ss,0.4814,0.4623


### Relative accuracies

In [19]:
il.reload(ut)
res = ut.compare_experiments(["tabnet_owntop1_comp_ss_sup_logits","lgbm_top1_ss_logits"],title='tabnetowntop1_vs_lgbm',save=False)

Getting aliases...


In [20]:
res

,match,Date,season,Div,HomeTeam,AwayTeam,FTHG,FTAG,label,prediction_lgbm_top1_ss_logits,...,home_lgbm_top1_ss_logits,away_lgbm_top1_ss_logits,prediction_tabnet_owntop1_comp_ss_sup_logits,draw_tabnet_owntop1_comp_ss_sup_logits,home_tabnet_owntop1_comp_ss_sup_logits,away_tabnet_owntop1_comp_ss_sup_logits,accurate_flag_lgbm_top1_ss_logits,rps_lgbm_top1_ss_logits,accurate_flag_tabnet_owntop1_comp_ss_sup_logits,rps_tabnet_owntop1_comp_ss_sup_logits
0,15751,2020-01-01,T19-20,E0,Brighton,Chelsea,1,1,0.0,2,...,0.2835,0.4594,0.0,0.5139,0.2421,0.2439,0,0.381474,1.0,0.147915
1,15757,2020-01-01,T19-20,E0,Norwich,Crystal Palace,1,1,0.0,2,...,0.2340,0.4634,0.0,0.6307,0.2160,0.1532,0,0.350553,1.0,0.079942
2,15754,2020-01-01,T19-20,E0,Southampton,Tottenham,1,0,1.0,1,...,0.3702,0.2678,1.0,0.2356,0.7244,0.0400,1,0.101380,1.0,0.028554
3,15753,2020-01-01,T19-20,E0,Newcastle,Leicester,0,3,2.0,2,...,0.1720,0.6137,2.0,0.2081,0.0677,0.7242,1,0.097576,1.0,0.059686
4,15756,2020-01-01,T19-20,E0,Man City,Everton,2,1,1.0,1,...,0.7817,0.0623,1.0,0.2329,0.7463,0.0208,1,0.014109,1.0,0.027338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3238,26489,2021-05-23,T20-21,F1,St Etienne,Dijon,0,1,2.0,1,...,0.6681,0.0960,1.0,0.2958,0.5845,0.1196,0,0.436432,0.0,0.431213
3239,26490,2021-05-23,T20-21,F1,Strasbourg,Lorient,1,1,0.0,1,...,0.4648,0.2517,2.0,0.3529,0.2875,0.3596,0,0.288363,0.0,0.274025
3240,44888,2021-05-23,T20-21,SP1,Sevilla,Alaves,1,0,1.0,1,...,0.8088,0.0471,1.0,0.2415,0.6932,0.0653,1,0.011492,1.0,0.031293
3241,26481,2021-05-23,T20-21,F1,Angers,Lille,1,2,2.0,2,...,0.2058,0.5717,2.0,0.2813,0.2454,0.4734,1,0.116474,1.0,0.178271


In [24]:
def read_data(path,dtypes=None):
    return pd.read_csv(path,sep=';',decimal=',',parse_dates=['Date'],
                       date_format="%d/%m/%Y",dtype=dtypes)

dtypes = ut.load_json("F:\\TFG\\" + "datasets/raw_datasets/dtypes.json")
datalake = read_data("F:\TFG\datasets\\raw_datasets\datalake.csv",dtypes)

In [28]:
res

,match,Date,season,Div,HomeTeam,AwayTeam,FTHG,FTAG,label,prediction_lgbm_top1_ss_logits,...,home_lgbm_top1_ss_logits,away_lgbm_top1_ss_logits,prediction_tabnet_owntop1_comp_ss_sup_logits,draw_tabnet_owntop1_comp_ss_sup_logits,home_tabnet_owntop1_comp_ss_sup_logits,away_tabnet_owntop1_comp_ss_sup_logits,accurate_flag_lgbm_top1_ss_logits,rps_lgbm_top1_ss_logits,accurate_flag_tabnet_owntop1_comp_ss_sup_logits,rps_tabnet_owntop1_comp_ss_sup_logits
0,15751,2020-01-01,T19-20,E0,Brighton,Chelsea,1,1,0.0,2,...,0.2835,0.4594,0.0,0.5139,0.2421,0.2439,0,0.381474,1.0,0.147915
1,15757,2020-01-01,T19-20,E0,Norwich,Crystal Palace,1,1,0.0,2,...,0.2340,0.4634,0.0,0.6307,0.2160,0.1532,0,0.350553,1.0,0.079942
2,15754,2020-01-01,T19-20,E0,Southampton,Tottenham,1,0,1.0,1,...,0.3702,0.2678,1.0,0.2356,0.7244,0.0400,1,0.101380,1.0,0.028554
3,15753,2020-01-01,T19-20,E0,Newcastle,Leicester,0,3,2.0,2,...,0.1720,0.6137,2.0,0.2081,0.0677,0.7242,1,0.097576,1.0,0.059686
4,15756,2020-01-01,T19-20,E0,Man City,Everton,2,1,1.0,1,...,0.7817,0.0623,1.0,0.2329,0.7463,0.0208,1,0.014109,1.0,0.027338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3238,26489,2021-05-23,T20-21,F1,St Etienne,Dijon,0,1,2.0,1,...,0.6681,0.0960,1.0,0.2958,0.5845,0.1196,0,0.436432,0.0,0.431213
3239,26490,2021-05-23,T20-21,F1,Strasbourg,Lorient,1,1,0.0,1,...,0.4648,0.2517,2.0,0.3529,0.2875,0.3596,0,0.288363,0.0,0.274025
3240,44888,2021-05-23,T20-21,SP1,Sevilla,Alaves,1,0,1.0,1,...,0.8088,0.0471,1.0,0.2415,0.6932,0.0653,1,0.011492,1.0,0.031293
3241,26481,2021-05-23,T20-21,F1,Angers,Lille,1,2,2.0,2,...,0.2058,0.5717,2.0,0.2813,0.2454,0.4734,1,0.116474,1.0,0.178271


In [36]:
res_bets = res.merge(datalake[["matchId","B365H","B365D","B365A"]],left_on="match",right_on="matchId",how="left").drop(columns="matchId")
res_bets["B365H"] = 1 / res_bets["B365H"]
res_bets["B365D"] = 1 / res_bets["B365D"]
res_bets["B365A"] = 1 / res_bets["B365A"]
ut.save_dataframe(res_bets,ut.SAVE_PATH,name_of_file="logits_bets",to_excel=True,sheet_name='logits_bets')
res_bets

,match,Date,season,Div,HomeTeam,AwayTeam,FTHG,FTAG,label,prediction_lgbm_top1_ss_logits,...,draw_tabnet_owntop1_comp_ss_sup_logits,home_tabnet_owntop1_comp_ss_sup_logits,away_tabnet_owntop1_comp_ss_sup_logits,accurate_flag_lgbm_top1_ss_logits,rps_lgbm_top1_ss_logits,accurate_flag_tabnet_owntop1_comp_ss_sup_logits,rps_tabnet_owntop1_comp_ss_sup_logits,B365H,B365D,B365A
0,15751,2020-01-01,T19-20,E0,Brighton,Chelsea,1,1,0.0,2,...,0.5139,0.2421,0.2439,0,0.381474,1.0,0.147915,0.277778,0.277778,0.512821
1,15757,2020-01-01,T19-20,E0,Norwich,Crystal Palace,1,1,0.0,2,...,0.6307,0.2160,0.1532,0,0.350553,1.0,0.079942,0.400000,0.294118,0.363636
2,15754,2020-01-01,T19-20,E0,Southampton,Tottenham,1,0,1.0,1,...,0.2356,0.7244,0.0400,1,0.101380,1.0,0.028554,0.303030,0.285714,0.476190
3,15753,2020-01-01,T19-20,E0,Newcastle,Leicester,0,3,2.0,2,...,0.2081,0.0677,0.7242,1,0.097576,1.0,0.059686,0.200000,0.263158,0.602410
4,15756,2020-01-01,T19-20,E0,Man City,Everton,2,1,1.0,1,...,0.2329,0.7463,0.0208,1,0.014109,1.0,0.027338,0.800000,0.153846,0.100000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3238,26489,2021-05-23,T20-21,F1,St Etienne,Dijon,0,1,2.0,1,...,0.2958,0.5845,0.1196,0,0.436432,0.0,0.431213,0.769231,0.181818,0.100000
3239,26490,2021-05-23,T20-21,F1,Strasbourg,Lorient,1,1,0.0,1,...,0.3529,0.2875,0.3596,0,0.288363,0.0,0.274025,0.384615,0.500000,0.230947
3240,44888,2021-05-23,T20-21,SP1,Sevilla,Alaves,1,0,1.0,1,...,0.2415,0.6932,0.0653,1,0.011492,1.0,0.031293,0.666667,0.230947,0.153846
3241,26481,2021-05-23,T20-21,F1,Angers,Lille,1,2,2.0,2,...,0.2813,0.2454,0.4734,1,0.116474,1.0,0.178271,0.100000,0.210526,0.751880


In [27]:
res.label.value_counts()

label
1.0    1338
2.0    1049
0.0     856
Name: count, dtype: int64

In [342]:
accurate_flag_cols = ut.filter_list(r"accurate_flag_\w+", ", ".join(res.columns))
res_acc = res[ ut.METADATA + accurate_flag_cols]
names = [ ut.filter_group("accurate_flag_(\w+)",col) for col in res.columns if ut.filter_group("accurate_flag_(\w+)",col)]
rel_test = { name:res_acc[f'accurate_flag_{name}'].mean().round(4) for name in names }

In [343]:
resume["rel_test"] = rel_test
resume.sort_values('rel_test',ascending=False)

,validation,abs_test,rel_test
tabnet_own_comp_ss,0.5327,0.5021,0.5021
own_original_comp_fac_ss,0.5347,0.4998,0.4998
own_original_simp_fac_ss,0.5312,0.4998,0.4998
tabnet_own_simp_ss_unsup,0.5266,0.4964,0.4964
tabnet_own_comp_ss_sup,0.5314,0.4941,0.4941
tabnet_own_simp_ss_sup,0.5298,0.4941,0.4941
tabnet_own_comp_rs_unsup,0.5220,0.4922,0.4922
own_original_comp_bal_fac_ss,0.4814,0.4623,0.4869
tabnet_paper_ss_unsup,0.5180,0.4857,0.4846
tabnet_paper_rs_unsup,0.5063,0.4807,0.4838


### RPS

In [344]:
res

,match,Date,season,Div,HomeTeam,AwayTeam,FTHG,FTAG,label,prediction_own_original_comp_bal_fac_ss,...,accurate_flag_own_original_comp_bal_fac_ss,accurate_flag_own_original_comp_fac_ss,accurate_flag_own_original_simp_fac_ss,accurate_flag_tabnet_own_comp_rs_unsup,accurate_flag_tabnet_own_comp_ss,accurate_flag_tabnet_own_comp_ss_sup,accurate_flag_tabnet_own_simp_ss_sup,accurate_flag_tabnet_own_simp_ss_unsup,accurate_flag_tabnet_paper_rs_unsup,accurate_flag_tabnet_paper_ss_unsup
0,15751,2020-01-01,T19-20,E0,Brighton,Chelsea,1,1,0,2,...,0,0,0,0,0,0,0,0,0,0
1,15757,2020-01-01,T19-20,E0,Norwich,Crystal Palace,1,1,0,2,...,0,0,0,0,0,0,0,0,0,0
2,15756,2020-01-01,T19-20,E0,Man City,Everton,2,1,1,1,...,1,1,1,1,1,1,1,1,1,1
3,15752,2020-01-01,T19-20,E0,Burnley,Aston Villa,1,2,2,0,...,0,0,0,0,0,0,0,0,0,0
4,15755,2020-01-01,T19-20,E0,Watford,Wolves,2,1,1,2,...,0,0,0,0,1,0,0,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2626,26489,2021-05-23,T20-21,F1,St Etienne,Dijon,0,1,2,1,...,0,0,0,0,0,0,0,0,0,0
2627,26487,2021-05-23,T20-21,F1,Reims,Bordeaux,1,2,2,0,...,0,0,0,0,0,0,0,0,0,1
2628,26486,2021-05-23,T20-21,F1,Nantes,Montpellier,1,2,2,0,...,0,0,0,0,0,0,0,0,0,0
2629,26488,2021-05-23,T20-21,F1,Rennes,Nimes,2,0,1,1,...,1,1,1,1,1,1,1,1,1,1


In [346]:
outcomes = res[['draw_tabnet_own_comp_ss','home_tabnet_own_comp_ss','away_tabnet_own_comp_ss']].values
labels = pd.get_dummies(res.label).astype(float).values

resta = abs(outcomes - labels)
resta

array([[0.7526, 0.3026, 0.45  ],
       [0.725 , 0.3305, 0.3945],
       [0.1584, 0.2485, 0.0901],
       ...,
       [0.2961, 0.4702, 0.7663],
       [0.2763, 0.4394, 0.1631],
       [0.7167, 0.4842, 0.2325]])

In [350]:
rps_dict = { name:ut.compute_rps(res,name)[0] for name in names }
rps_dict
resume['RPS'] = rps_dict
resume.sort_values('RPS')

,validation,abs_test,rel_test,RPS
tabnet_own_comp_ss,0.5327,0.5021,0.5021,0.1926
own_original_comp_fac_ss,0.5347,0.4998,0.4998,0.1931
tabnet_own_comp_ss_sup,0.5314,0.4941,0.4941,0.1934
tabnet_own_comp_rs_unsup,0.5220,0.4922,0.4922,0.1938
own_original_simp_fac_ss,0.5312,0.4998,0.4998,0.1945
tabnet_own_simp_ss_unsup,0.5266,0.4964,0.4964,0.1947
own_original_comp_bal_fac_ss,0.4814,0.4623,0.4869,0.1949
tabnet_own_simp_ss_sup,0.5298,0.4941,0.4941,0.1956
tabnet_paper_rs_unsup,0.5063,0.4807,0.4838,0.1974
tabnet_paper_ss_unsup,0.5180,0.4857,0.4846,0.1982


In [5]:
il.reload(ut)
ut.make_resume([r'(\w+)'],title='resume',save=True)

Getting aliases...


,validation,abs_test,RPS,rel_test
tabnet_owntop1_comp_ss_sup,0.6508,0.5919,0.1707,0.5919
lgbm_top1_ss,1.0428,0.5643,0.1746,0.5535
lgbm_top1_rs,1.0530,0.5692,0.1747,0.5581
tabnet_own_comp_ss,0.5327,0.5021,0.1926,0.5033
tabnet_own_comp_ss_sup,0.5341,0.4945,0.1932,0.4956
tabnet_own_comp_rs_unsup,0.5220,0.4922,0.1938,0.4929
own_original_simp_fac_ss,0.5312,0.4998,0.1945,0.5010
tabnet_own_simp_ss_unsup,0.5266,0.4964,0.1947,0.4975
own_original_comp_bal_fac_ss,0.4814,0.4623,0.1949,0.4875
tabnet_own_simp_ss_sup,0.5298,0.4941,0.1956,0.4952


In [48]:
df = pd.DataFrame({'c1':[1,0,1,0,1,0],'c2':[1,None,0,0,None,1]})
ind = df.c2.dropna().index
df['ceq'] = (df.c1.loc[ind]==df.c2[ind]).astype(int)
df

,c1,c2,ceq
0,1,1.0,1.0
1,0,NaN,NaN
2,1,0.0,0.0
3,0,0.0,1.0
4,1,NaN,NaN
5,0,1.0,0.0


DECIDIR QUE MODELOS LANZAR A ENTRENAR DE NUEVO
1. Preparar modelos con LightGBM
2. Preparar modelo Top1
3. Preparar mis modelos + atributos Top1
4. Investigar Embeddings
5. Investigar Scrapping

#### end